# Limpieza de datos corregida a partir de los resultados del Laboratorio 1

## 0. Instalación de librerías necesarias

In [ ]:
%pip install pandas numpy openpyxl matplotlib seaborn scikit-learn

## 1. Importación de librerías necesarias

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder

In [ ]:
print(f"Versión de pandas: {pd.__version__}")
print(f"Versión de numpy: {np.__version__}")

## **3. Exploración de datos**

### 3.1 Carga de datos original y copia

In [ ]:
df = pd.read_csv('Datos/Datos Lab 1.csv') # Importar los datos originales con pandas
df_original = df.copy() # Crear una copia del DataFrame original para referencia futura 

### 3.2 Mostrar primeras filas del DataFrame original

In [ ]:
df_original.head() # Mostrar las primeras filas del DataFrame original para inspección inicial

### 3.3 Mostrar información general del DataFrame original y el conteo de valores nulos

In [ ]:
print("==================================")
print("Información del DataFrame original:")
print("================================== \n")
print("Shape original: ", df_original.shape, "\n") # Mostrar la forma del DataFrame original para entender la cantidad de filas y columnas
df_original.info() # Mostrar información detallada del DataFrame original, incluyendo tipos de datos y conteo de valores no nulos

> ### Valores faltantes por variable

- **Age**: Tipo: numérica continua | Nulos: 68
- **Weight (kg)**: Tipo: numérica continua | Nulos: 73
- **Height (m)**: Tipo: numérica continua | Nulos: 61
- **BMI**: Tipo: numérica continua | Nulos: 53
- **Abdominal Circumference (cm)**: Tipo: numérica continua | Nulos: 61
- **Total Cholesterol (mg/dL)**: Tipo: numérica continua | Nulos: 68
- **HDL (mg/dL)**: Tipo: numérica continua | Nulos: 82
- **Fasting Blood Sugar (mg/dL)**: Tipo: numérica continua | Nulos: 54
- **Height (cm)**: Tipo: numérica continua | Nulos: 68
- **Waist-to-Height Ratio**: Tipo: numérica continua | Nulos: 76
- **Systolic BP**: Tipo: numérica continua | Nulos: 61
- **Diastolic BP**: Tipo: numérica continua | Nulos: 85
- **Estimated LDL (mg/dL)**: Tipo: numérica continua | Nulos: 57
- **CVD Risk Score**: Tipo: numérica continua | Nulos: 29

### 3.4 Importación del diccionario de datos para análisis detallado de cada variable

In [ ]:
diccionario = pd.read_excel('Datos/DiccPacientes.xlsx') # Importar el diccionario de datos con pandas
pd.set_option('display.max_colwidth', None) # Configurar pandas para mostrar el contenido completo de las celdas del diccionario
diccionario

> ### Diccionario de datos

A partir del diccionario se identificaron los siguientes aspectos relevantes:

**Columnas redundantes detectadas:**
- `Height (m)` y `Height (cm)` representan la misma variable en distintas unidades
- `Blood Pressure (mmHg)` ya está separada en `Systolic BP` y `Diastolic BP`
- `CVD Risk Level` es la versión categórica de `CVD Risk Score` (target leakage)

**Columnas no relevantes para el modelo:**
- `Patient ID` es un identificador único, no aporta información predictiva
- `Date of Service` es la fecha de atención, no tiene relación con el riesgo cardiovascular

**Variables objetivo:**
- `CVD Risk Score` es la variable que vamos a predecir (numérica continua)
- `CVD Risk Level` es su versión categórica, debe excluirse del modelo

### 3.5 Descripción estadistica de cada variable

In [ ]:
df.describe() # Mostrar estadísticas descriptivas de las variables numéricas para entender su distribución y detectar posibles anomalías

> ### Estadísticas descriptivas

A partir de `df.describe()` se identificaron los siguientes valores fuera de rango:

**Valores imposibles (errores de datos):**

- **BMI** | Mínimo: 4.31
Fisiológicamente imposible. El mínimo registrado mundialmente es ~7.5.
Se convertirá a NaN.

- **Total Cholesterol (mg/dL)** | Mínimo: -1.25
Fisiológicamente imposible. El colesterol no puede ser negativo.
Se convertirá a NaN.

- **HDL (mg/dL)** | Mínimo: 0.008
Fisiológicamente imposible. HDL no puede ser cercano a cero.
Se convertirá a NaN todo valor menor a 10.

- **Estimated LDL (mg/dL)** | Mínimo: -92.05
Fisiológicamente imposible. LDL no puede ser negativo.
Se convertirá a NaN.

- **CVD Risk Score** | Mínimo: -20.05, Máximo: 114.98
Fuera del rango válido [0, 100].
Se convertirán a NaN los valores fuera de este rango.

- **Weight (kg)** | Mínimo: 13.26
Imposible para un adulto. Se convertirá a NaN todo valor menor a 20 kg.

- **Fasting Blood Sugar (mg/dL)** | Mínimo: 15.30
Imposible fisiológicamente. Se convertirá a NaN todo valor menor a 30.

**Outliers extremos pero posibles:**

- **Age** | Mínimo: 6.13
Inusual para un estudio cardiovascular pero no imposible. Se conserva.

- **BMI** | Máximo: 53
Obesidad mórbida extrema, posible. Se conserva.

- **CVD Risk Score** | Máximo dentro de rango: ~60
Pacientes de alto riesgo, válido. Se conserva.

### 3.6 Revisar formato de categoricas

In [ ]:
categoricas = df.select_dtypes(include=['object']).columns # Identificar las columnas categóricas en el DataFrame

for i in categoricas:
    display(df[i].value_counts()) # Mostrar el conteo de valores únicos para cada columna categórica para detectar posibles errores de formato o inconsistencias

> ### Variables categóricas — Value Counts

A partir de `df.value_counts()` se analizaron las variables categóricas del dataset:

- **Patient ID:** Hay IDs repetidos, con algunos apareciendo hasta 3 veces. 
Esto sugiere registros duplicados que se analizarán a continuación para 
determinar si corresponden a visitas distintas o registros mal ingresados.

- **Date of Service:**
Se detectaron múltiples formatos de fecha inconsistentes. De todas maneras, esta columna no tiene relación con el riesgo cardiovascular de un paciente  por lo que se eliminará independientemente del formato. Algunos ejemplos de formatos encontrados:
    - 09-20-2023
    - December 05, 2025
    - 08 Mar 22
    - 2021-05-01



- **Sex:** Dos categorías balanceadas: M (821) y F (818). Sin inconsistencias.

- **Blood Pressure (mmHg):** Esta columna es redundante ya que su información está separada en 
Systolic BP y Diastolic BP.

- **Smoking Status:** Dos categorías balanceadas: Y (850) y N (789). Sin inconsistencias.

- **Diabetes Status:** Dos categorías balanceadas: N (821) e Y (818). Sin inconsistencias.

- **Physical Activity Level:** Tres categorías bien distribuidas: High (582), Moderate (537), Low (520). 
Sin inconsistencias.

- **Family History of CVD:** Dos categorías balanceadas: N (820) e Y (819). Sin inconsistencias.

- **Blood Pressure Category:** Cuatro categorías con distribución desbalanceada. Hypertension Stage 2 
domina con 680 registros mientras Elevated solo tiene 111. Sin inconsistencias 
en los valores.

- **CVD Risk Level:** Tres categorías desbalanceadas: HIGH (793), INTERMEDIARY (616), LOW (230). 
Esta columna es target leakage ya que es la versión categórica de CVD Risk 
Score, la variable objetivo.

### 3.7 Análisis extenso de duplicados y variables categoricas para determinar acciones de limpieza

In [ ]:
# Total de duplicados exactos
print(f"Total filas: {len(df)}")
print(f"Duplicados exactos (todas las columnas): {df.duplicated().sum()}")

# IDs repetidos
ids_repetidos = df[df['Patient ID'].duplicated(keep=False)]
print(f"IDs únicos repetidos: {ids_repetidos['Patient ID'].nunique()}")
print(f"Filas con ID repetido: {len(ids_repetidos)}")

# Filas con ID repetido pero datos distintos
no_exactos = ids_repetidos[~ids_repetidos.duplicated(keep=False)]
print(f"Filas con ID repetido pero datos distintos: {len(no_exactos)}")

# Mostrar ejemplos
print("\nEjemplos de registros con ID repetido y datos distintos:")
print(no_exactos.sort_values('Patient ID')[
    ['Patient ID', 'Date of Service', 'Age', 'Weight (kg)', 'CVD Risk Score']
].head(20))

> ### Duplicados - Casos específicos

Se identificaron tres tipos de registros problemáticos:

**Tipo 1 — Duplicados exactos: 151 filas**
Filas completamente idénticas en todas las columnas. No aportan información
adicional al modelo. Se eliminarán con `drop_duplicates()`.

**Tipo 2 — Mismo ID, misma fecha, CVD Risk Score imposible: parte de las 132 filas**
Mismo paciente y misma visita pero con un CVD Risk Score fuera del rango
válido [0, 100], como scores negativos. El valor imposible se convertirá
a NaN primero, luego `drop_duplicates()` eliminará la fila duplicada
conservando la que tiene el score válido.

**Tipo 3 — Mismo ID, misma fecha, CVD Risk Score distinto pero ambos válidos: resto de las 132 filas**
Mismo paciente y misma visita pero con dos scores válidos distintos, como
Axab9332 con scores 15.27 y 23.43, o BqZp2317 con 16.87 y 7.05. No hay
forma de determinar cuál es el correcto, por lo que se conservará el primer
registro encontrado usando `drop_duplicates(subset=['Patient ID'], keep='first')`.

**Orden de limpieza definido:**
1. Convertir CVD Risk Score imposible a NaN
2. Eliminar duplicados exactos con `drop_duplicates()`
3. Eliminar registros con mismo ID conservando el primero con
   `drop_duplicates(subset=['Patient ID'], keep='first')`
4. Eliminar filas sin CVD Risk Score con `dropna(subset=['CVD Risk Score'])`

### 3.8 Graficos estadisticos

#### 3.8.1 Boxplots de variables numéricas para identificar outliers y rangos válidos

In [ ]:
# Se incluyen unicamente variables numericas para los boxplots
numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns

fig, axes = plt.subplots(4, 4, figsize=(16, 12))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    if i < len(axes):
        df.boxplot(column=col, ax=axes[i])
        axes[i].set_title(col)

plt.tight_layout()
plt.show()

> ### Observaciones — Boxplots

- **Age:**
Distribución relativamente simétrica con mediana alrededor de 46 años.
Se observan outliers en el extremo inferior (~6 años) que son inconsistentes
con el contexto del estudio, el cual se enfoca en población adulta. Se 
convertirán a NaN los valores menores a 18 años.

- **Weight (kg):**
Distribución ligeramente sesgada hacia arriba con la mayoría de valores
entre 60 y 120 kg. No se observan outliers estadísticos en el boxplot,
sin embargo en el análisis del describe se identificó un mínimo de 13 kg
que es imposible para un adulto y se tratará en la fase de limpieza.

- **Height (m):**
Distribución simétrica y sin outliers extremos. Los valores están dentro
de rangos fisiológicamente posibles.

- **BMI:**
Distribución ligeramente sesgada hacia arriba. Se observan outliers en el
extremo inferior (~4) que son fisiológicamente imposibles y se convertirán
a NaN. El máximo (~50) es alto pero posible en obesidad mórbida.

- **Abdominal Circumference (cm):**
Distribución simétrica sin outliers extremos. Todos los valores parecen
estar dentro de rangos posibles.

- **Total Cholesterol (mg/dL):**
Distribución simétrica. Se observa un outlier en el extremo inferior
cercano a 0 que es fisiológicamente imposible y se convertirá a NaN.

- **HDL (mg/dL):**
Distribución simétrica. Se observan outliers en el extremo inferior
cercanos a 0 que son fisiológicamente imposibles y se convertirán a NaN.

- **Fasting Blood Sugar (mg/dL):**
Distribución ligeramente sesgada hacia arriba. Se observan outliers en
el extremo inferior (~15) que son fisiológicamente imposibles y se
convertirán a NaN.

- **Height (cm):**
Redundante con Height (m). Se eliminará esta columna independientemente
de sus valores.

- **Waist-to-Height Ratio:**
Distribución simétrica. Se observa un outlier superior (~0.8) que es
alto pero posible. Sin valores imposibles.

- **Systolic BP:**
Distribución simétrica. Se observan outliers en ambos extremos pero
dentro de rangos posibles (~50 y ~200).

- **Diastolic BP:**
Distribución simétrica. Se observan outliers en el extremo inferior
(~35) y superior (~130) pero dentro de rangos posibles.

- **Estimated LDL (mg/dL):**
Se observan outliers claros en el extremo inferior con valores negativos
(~-100) que son fisiológicamente imposibles y se convertirán a NaN.
El máximo (~300) es alto pero posible.

- **CVD Risk Score:**
Se observan outliers en ambos extremos con valores negativos (~-20) y
superiores a 100 (~115) que están fuera del rango válido [0, 100] y se
convertirán a NaN. La distribución está sesgada hacia valores bajos con
mediana alrededor de 17.

> ### Rangos clínicamente imposibles verificados

- **Age:**
    - Mínimo posible: 18 años (contexto de estudio cardiovascular en adultos).
    - Valor en dataset: 6.13 -> se convertirá a NaN.

- **Weight (kg):**
    - Mínimo posible para un adulto: ~30 kg.
    - Valor en dataset: 13.26 -> se convertirá a NaN.

- **BMI:**
    - Mínimo registrado con supervivencia médica: ~7.8 kg/m².
    - En contexto de estudio cardiovascular ambulatorio: mínimo razonable ~10.
    - Valor en dataset: 4.31 -> se convertirá a NaN.

- **Total Cholesterol (mg/dL):**
    - Fisiológicamente imposible ser negativo o menor a 50 mg/dL.
    - Valor en dataset: -1.25 -> se convertirá a NaN.

- **HDL (mg/dL)**
    - Mínimo fisiológico documentado: ~5 mg/dL en enfermedades genéticas raras.
    - En contexto clínico práctico: valores menores a 10 son imposibles.
    - Valor en dataset: 0.008 -> se convertirá a NaN todo valor menor a 10.

- **Fasting Blood Sugar (mg/dL)**
    - Por debajo de 40 mg/dL es hipoglucemia severa potencialmente fatal.
    - Por debajo de 15 mg/dL es incompatible con la consciencia.
    - Valor en dataset: 15.30 -> se convertirá a NaN todo valor menor a 40.

- **Estimated LDL (mg/dL)**
    - Fisiológicamente imposible ser negativo.
    - Valor en dataset: -92.05 -> se convertirá a NaN.

- **Systolic BP (mmHg)**
    - Mínimo para perfusión de órganos vitales: ~70 mmHg.
    - Valor en dataset: ~49.9 -> se convertirá a NaN todo valor menor a 70.

- **Diastolic BP (mmHg)**
    - Mínimo fisiológico razonable: ~40 mmHg.
    - Valor en dataset: ~31.7 -> se convertirá a NaN todo valor menor a 40.

- **CVD Risk Score**
    - Rango válido definido: [0, 100].
    - Valores en dataset: -20.05 y 114.98 -> se convertirán a NaN.

#### 3.8.2 Histogramas de variables numéricas para analizar distribuciones y detectar sesgos o anomalías

In [ ]:
numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns

fig, axes = plt.subplots(4, 4, figsize=(16, 12))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    if i < len(axes):
        axes[i].hist(df[col].dropna(), bins=30, edgecolor='black')
        axes[i].set_title(col)
        axes[i].set_xlabel('Valor')
        axes[i].set_ylabel('Frecuencia')

plt.tight_layout()
plt.show()

> ### Observaciones — Histogramas

- **Age:**
Distribución aproximadamente normal y simétrica centrada alrededor de 
los 40-50 años. Confirma lo visto en el boxplot.

- **Weight (kg):**
Distribución aproximadamente normal con leve sesgo hacia la derecha. 
La mayoría de pacientes pesa entre 60 y 120 kg.

- **Height (m):**
Distribución normal y simétrica centrada alrededor de 1.75 m. 
Sin anomalías visibles.

- **BMI:**
Distribución normal con leve sesgo hacia la derecha. La mayoría 
de pacientes tiene BMI entre 20 y 40.

- **Abdominal Circumference (cm):**
Distribución aproximadamente normal centrada alrededor de 90 cm 
con leve sesgo hacia la derecha.

- **Total Cholesterol (mg/dL):**
Distribución aproximadamente normal centrada alrededor de 200 mg/dL. 
Se confirma la barra aislada cerca de 0 correspondiente a los valores 
imposibles ya identificados.

- **HDL (mg/dL):**
Distribución con sesgo hacia la derecha, la mayoría de valores 
concentrados entre 30 y 80 mg/dL. Se confirma la barra aislada 
cercana a 0 de los valores imposibles ya identificados.

- **Fasting Blood Sugar (mg/dL):**
Distribución aproximadamente normal centrada alrededor de 115 mg/dL 
con leve sesgo hacia la derecha. Se confirma una barra aislada 
cerca de 15 correspondiente a valores imposibles ya identificados.

- **Height (cm):**
Distribución normal centrada alrededor de 175 cm. Redundante con 
Height (m), se eliminará.

- **Waist-to-Height Ratio:**
Distribución normal y simétrica centrada alrededor de 0.5. 
Sin anomalías visibles.

- **Systolic BP:**
Distribución aproximadamente normal centrada alrededor de 125 mmHg. 
Sin anomalías visibles.

- **Diastolic BP:**
Distribución aproximadamente normal centrada alrededor de 80 mmHg 
con leve sesgo hacia la izquierda. Se confirma una barra aislada 
cerca de 35 correspondiente a valores bajos ya identificados.

- **Estimated LDL (mg/dL):**
Distribución aproximadamente normal centrada alrededor de 110 mg/dL. 
Se confirma claramente una barra aislada en valores negativos 
correspondiente a los valores imposibles ya identificados.

- **CVD Risk Score:**
Distribución con fuerte sesgo hacia la derecha, con la gran mayoría 
de pacientes concentrados entre 10 y 30 puntos. Se confirman barras 
aisladas en valores negativos correspondientes a los valores 
imposibles ya identificados. Importante tener en cuenta este sesgo 
al evaluar el modelo.

#### 3.8.3 Scatter plots de variables numéricas para analizar relaciones entre ellas y detectar patrones o anomalías

In [ ]:
numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns
target = 'CVD Risk Score'

cols_to_plot = [col for col in numeric_cols if col != target]

fig, axes = plt.subplots(4, 4, figsize=(16, 12))
axes = axes.flatten()

for i, col in enumerate(cols_to_plot):
    if i < len(axes):
        axes[i].scatter(df[col], df[target], alpha=0.3, s=10)
        axes[i].set_xlabel(col)
        axes[i].set_ylabel('CVD Risk Score')
        axes[i].set_title(f'{col} vs CVD Risk Score')

plt.tight_layout()
plt.show()

> ### Observaciones — Scatter Plots vs CVD Risk Score

- **Age:**
No se observa una relación lineal clara. Los puntos están distribuidos 
de forma relativamente uniforme en todos los rangos de edad, aunque 
hay una leve tendencia a scores más altos en edades medias.

- **Weight (kg):**
No se observa relación lineal con el CVD Risk Score. Los puntos 
están dispersos uniformemente sin un patrón claro.

- **Height (m):**
No se observa ninguna relación con el CVD Risk Score. 
Distribución completamente aleatoria.

- **BMI:**
Leve tendencia positiva, a mayor BMI ligeramente mayor CVD Risk Score, 
pero la relación es muy débil. Confirma la correlación baja de 0.13 
vista en el análisis previo.

- **Abdominal Circumference (cm):**
Similar al BMI, leve tendencia positiva pero muy débil. 
Sin patrón lineal claro.

- **Total Cholesterol (mg/dL):**
No se observa relación lineal con el CVD Risk Score. 
Distribución uniforme en todos los rangos.

- **HDL (mg/dL):**
No se observa relación lineal clara. Los puntos están 
dispersos uniformemente.

- **Fasting Blood Sugar (mg/dL):**
No se observa relación lineal con el CVD Risk Score. 
Distribución completamente aleatoria.

- **Height (cm):**
Redundante con Height (m). Se eliminará. No se observa 
relación con el CVD Risk Score.

- **Waist-to-Height Ratio:**
No se observa relación lineal clara con el CVD Risk Score.

- **Systolic BP:**
Leve tendencia positiva, a mayor presión sistólica ligeramente 
mayor CVD Risk Score, pero la relación es muy débil.

- **Diastolic BP:**
No se observa relación lineal clara con el CVD Risk Score.

- **Estimated LDL (mg/dL):**
No se observa relación lineal con el CVD Risk Score. 
Distribución uniforme en todos los rangos.

> ### Conclusión general de los scatter plots
Ninguna variable numérica muestra una relación lineal fuerte con 
CVD Risk Score. Las correlaciones son muy débiles en todos los casos. 
Esto sugiere que las relaciones pueden ser no lineales o que el poder 
predictivo está más en las variables categóricas y en combinaciones 
de variables que en variables individuales. Esto justifica explorar 
regresión polinomial y features derivadas en el lab 2.

#### 3.8.4 Gráficos de barras para variables categóricas para analizar distribuciones y detectar desbalances o anomalías

In [ ]:
categoricas = df.select_dtypes(include=['object']).columns

fig, axes = plt.subplots(3, 3, figsize=(14, 10))
axes = axes.flatten()

for i, col in enumerate(categoricas):
    if i < len(axes):
        df[col].value_counts().plot(kind='bar', ax=axes[i])
        axes[i].set_title(col)
        axes[i].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

> #### Observaciones — Gráficas de barras categóricas

- **Patient ID y Date of Service:**
Demasiados valores únicos para ser útiles como gráficas de barras. 
Confirma que estas columnas se eliminarán.

- **Blood Pressure (mmHg):**
Al ser una combinación de dos valores numéricos en formato texto 
como "127/84", genera demasiados valores únicos y la gráfica es 
ilegible. Confirma que esta columna se eliminará y se usarán 
Systolic BP y Diastolic BP que ya están separadas.

- **Sex:**
Dos categorías perfectamente balanceadas: M (821) y F (818). 
Sin inconsistencias.

- **Smoking Status:**
Dos categorías balanceadas: Y (850) y N (789). Sin inconsistencias.

- **Diabetes Status:**
Dos categorías balanceadas: N (821) e Y (818). Sin inconsistencias.

- **Physical Activity Level:**
Tres categorías bien distribuidas: High (582), Moderate (537), 
Low (520). Sin inconsistencias.

- **Family History of CVD:**
Dos categorías balanceadas: N (820) e Y (819). Sin inconsistencias.

- **Blood Pressure Category:**
Cuatro categorías desbalanceadas. Hypertension Stage 2 domina 
con ~680 registros mientras Elevated solo tiene ~111. 
Sin inconsistencias en los valores.

#### Matriz de correlación para variables numéricas para analizar relaciones entre ellas y detectar multicolinealidad o patrones

In [ ]:
# numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns

# plt.figure(figsize=(14, 10))
# sns.heatmap(df[numeric_cols].corr(), annot=True, cmap='coolwarm', 
#             center=0, fmt='.2f')
# plt.title('Matriz de Correlación — Variables Numéricas')
# plt.tight_layout()
# plt.show()

#### 3.8.5 Matriz de correlación para variables categóricas para analizar relaciones entre ellas y detectar patrones o desbalances

In [ ]:
from sklearn.preprocessing import LabelEncoder

df_encoded = df.copy()
categoricas = df.select_dtypes(include=['object']).columns
le = LabelEncoder()
for col in categoricas:
    df_encoded[col] = le.fit_transform(df_encoded[col].astype(str))

plt.figure(figsize=(16, 12))
sns.heatmap(df_encoded.corr(), annot=True, cmap='coolwarm', 
            center=0, fmt='.2f')
plt.title('Matriz de Correlación — Todas las Variables')
plt.tight_layout()
plt.show()

> #### Observaciones — Matriz de Correlación

**Correlaciones con CVD Risk Score (variable objetivo)**
Ninguna variable muestra correlación fuerte con el target. 
Las más relevantes son:
- BMI: 0.13 (más alta, pero muy débil)
- Diabetes Status: 0.15
- Systolic BP: 0.09
- Age: -0.00 (prácticamente nula)

Esto confirma lo visto en los scatter plots: ninguna variable 
por sí sola predice bien el CVD Risk Score, lo que justifica 
explorar combinaciones de variables y relaciones no lineales 
en el lab 2.

**Multicolinealidad detectada**
- Height (m) y Height (cm): 0.92 -> redundantes, se eliminará Height (cm)
- Total Cholesterol y Estimated LDL: 0.85 -> alta correlación, se evaluará si eliminar una en el pipeline
- Abdominal Circumference y Waist-to-Height Ratio: 0.84 -> alta correlación, se evaluará si eliminar una en el pipeline
- Weight y BMI: 0.58 -> correlación moderada esperada
- Blood Pressure (mmHg) y Systolic BP: 0.29 -> confirma redundancia

**Columnas a eliminar confirmadas**
- CVD Risk Level: correlación de 0.00 con CVD Risk Score lo cual  es sospechoso dado que debería derivarse de él. Confirma que  tiene target leakage y debe eliminarse.
- Patient ID y Date of Service: correlaciones cercanas a 0 con  todo, confirma que no aportan información predictiva.
- Height (cm): correlación de 0.92 con Height (m), confirma  que es redundante.
- Blood Pressure (mmHg): correlación de 0.29 con Systolic BP, confirma que es redundante con Systolic BP y Diastolic BP.

## 4. Resumen final de hallazgos y decisiones para la limpieza de datos

### 4.1 Hallazgos de la exploración

**Estructura del dataset**
El dataset original tiene 1639 filas y 24 columnas. De las 24 columnas,
14 son numéricas y 10 son categóricas.

**Valores faltantes**
Las columnas con mayor cantidad de nulos son Diastolic BP (85), 
HDL (82), Waist-to-Height Ratio (76), Weight (73) y Age (68). 
La variable objetivo CVD Risk Score tiene 29 nulos.

**Valores imposibles detectados**
A partir del describe y los boxplots se identificaron valores 
fisiológicamente imposibles en las siguientes variables:
- Age: mínimo 6.13 años (estudio de adultos)
- Weight: mínimo 13.26 kg (imposible para un adulto)
- BMI: mínimo 4.31 (mínimo de supervivencia documentado es ~7.8)
- Total Cholesterol: mínimo -1.25 mg/dL (imposible ser negativo)
- HDL: mínimo 0.008 mg/dL (imposible ser cercano a cero)
- Fasting Blood Sugar: mínimo 15.30 mg/dL (incompatible con la consciencia)
- Estimated LDL: mínimo -92.05 mg/dL (imposible ser negativo)
- Systolic BP: mínimo 49.91 mmHg (mínimo viable es ~70)
- Diastolic BP: mínimo 31.72 mmHg (mínimo viable es ~40)
- CVD Risk Score: mínimo -20.05 y máximo 114.98 (rango válido 0-100)

**Duplicados**
Se identificaron tres tipos de registros problemáticos:
- 151 filas completamente duplicadas
- Filas con mismo ID y score imposible
- 132 filas con mismo ID y scores válidos pero distintos

**Relaciones con el target**
Ninguna variable muestra correlación fuerte con CVD Risk Score.
Las más altas son Diabetes Status (0.15) y BMI (0.13). Los scatter
plots confirman que las relaciones son débiles y posiblemente no 
lineales, lo que justifica explorar regresión polinomial en el lab 2.

**Multicolinealidad**
- Height (m) y Height (cm): 0.92
- Total Cholesterol y Estimated LDL: 0.85
- Abdominal Circumference y Waist-to-Height Ratio: 0.84
- Weight y BMI: 0.58

---

### 4.2 Decisiones de limpieza

**Columnas a eliminar**
- Patient ID: identificador sin valor predictivo
- Date of Service: fecha sin relación con el riesgo cardiovascular
- Blood Pressure (mmHg): redundante con Systolic BP y Diastolic BP
- CVD Risk Level: target leakage, versión categórica del score
- Height (cm): redundante con Height (m)

**Valores imposibles a convertir a NaN**
- Age < 18
- Weight < 30 kg
- BMI < 10
- Total Cholesterol < 50 mg/dL
- HDL < 10 mg/dL
- Fasting Blood Sugar < 40 mg/dL
- Estimated LDL < 0 mg/dL
- Systolic BP < 70 mmHg
- Diastolic BP < 40 mmHg
- CVD Risk Score < 0 o > 100

**Orden de limpieza**
1. Eliminar columnas irrelevantes
2. Convertir valores imposibles a NaN
3. Eliminar duplicados exactos con drop_duplicates()
4. Eliminar registros con mismo ID conservando el primero con
   drop_duplicates(subset=['Patient ID'], keep='first')
5. Eliminar filas sin CVD Risk Score con dropna(subset=['CVD Risk Score'])

**Decisiones pendientes para el pipeline después del split**
- Imputación de nulos restantes con mediana o fórmulas médicas
- Tratamiento de multicolinealidad entre Cholesterol/LDL 
  y Abdominal Circumference/Waist-to-Height Ratio
- Encoding de variables categóricas con OneHotEncoder
- Escalamiento de variables numéricas

## 5. Limpieza de datos

### 5.1 Eliminar columnas irrelevantes
- Patient ID
- Date of Service
- Blood Pressure (mmHg)
- CVD Risk Level (Categórica del target)
- Height (cm)

In [ ]:
cols_eliminar = ['Date of Service', 'Blood Pressure (mmHg)', 'CVD Risk Level', 'Height (cm)'] # Aun no se elimina Patient ID para poder eliminar duplicados por ID posteriormente

df = df.drop(columns=cols_eliminar)
print(f"Columnas eliminadas: {cols_eliminar}")
print(f"Shape después de eliminar columnas: {df.shape}")

### 5.2 Convertir valores imposibles a NaN

In [ ]:
df.loc[df['Age'] < 18, 'Age'] = np.nan
df.loc[df['Weight (kg)'] < 30, 'Weight (kg)'] = np.nan
df.loc[df['BMI'] < 10, 'BMI'] = np.nan
df.loc[df['Total Cholesterol (mg/dL)'] < 50, 'Total Cholesterol (mg/dL)'] = np.nan
df.loc[df['HDL (mg/dL)'] < 10, 'HDL (mg/dL)'] = np.nan
df.loc[df['Fasting Blood Sugar (mg/dL)'] < 40, 'Fasting Blood Sugar (mg/dL)'] = np.nan
df.loc[df['Estimated LDL (mg/dL)'] < 0, 'Estimated LDL (mg/dL)'] = np.nan
df.loc[df['Systolic BP'] < 70, 'Systolic BP'] = np.nan
df.loc[df['Diastolic BP'] < 40, 'Diastolic BP'] = np.nan
df.loc[df['CVD Risk Score'] < 0, 'CVD Risk Score'] = np.nan
df.loc[df['CVD Risk Score'] > 100, 'CVD Risk Score'] = np.nan

print("Nulos después de convertir valores imposibles:")
print(df.isnull().sum())
print(f"\nShape: {df.shape}")

### 5.3 Eliminar duplicados exactos

In [ ]:
antes = len(df)
df = df.drop_duplicates()
print(f"Filas eliminadas por duplicados exactos: {antes - len(df)}")
print(f"Shape después de eliminar duplicados exactos: {df.shape}")

### 5.4 Eliminar registros con mismo ID conservando el primero

In [ ]:
antes = len(df)
df = df.drop_duplicates(subset=['Patient ID'], keep='first')
print(f"Filas eliminadas por ID duplicado: {antes - len(df)}")
print(f"Shape después de eliminar IDs duplicados: {df.shape}")

# Ya habiendome quedado con el primer registro de cada ID, ahora si puedo eliminar la columna de Patient ID
df = df.drop(columns=['Patient ID'])
print("Columna 'Patient ID' eliminada.")
print(f"Shape final del DataFrame limpio: {df.shape}")

### 5.5 Eliminar filas sin CVD Risk Score

In [ ]:
antes = len(df)
df = df.dropna(subset=['CVD Risk Score'])
print(f"Filas eliminadas por CVD Risk Score nulo: {antes - len(df)}")
print(f"Shape final: {df.shape}")